In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('Movies.csv', encoding='latin-1')

In [4]:
print("--- First 5 Rows ---")
print(df.head())

--- First 5 Rows ---
                                 Name    Year Duration            Genre  \
0                                         NaN      NaN            Drama   
1  #Gadhvi (He thought he was Gandhi)  (2019)  109 min            Drama   
2                         #Homecoming  (2021)   90 min   Drama, Musical   
3                             #Yaaram  (2019)  110 min  Comedy, Romance   
4                   ...And Once Again  (2010)  105 min            Drama   

   Rating Votes            Director       Actor 1             Actor 2  \
0     NaN   NaN       J.S. Randhawa      Manmauji              Birbal   
1     7.0     8       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande   
2     NaN   NaN  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur   
3     4.4    35          Ovais Khan       Prateik          Ishita Raj   
4     NaN   NaN        Amol Palekar  Rajat Kapoor  Rituparna Sengupta   

           Actor 3  
0  Rajendra Bhatia  
1    Arvind Jangid  
2       Roy Angana  
3  Si

In [5]:
print("\n--- Dataset Summary ---")
print(df.info())


--- Dataset Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  str    
 1   Year      14981 non-null  str    
 2   Duration  7240 non-null   str    
 3   Genre     13632 non-null  str    
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   str    
 6   Director  14984 non-null  str    
 7   Actor 1   13892 non-null  str    
 8   Actor 2   13125 non-null  str    
 9   Actor 3   12365 non-null  str    
dtypes: float64(1), str(9)
memory usage: 2.3 MB
None


In [6]:
print("\n--- Missing Values Count ---")
print(df.isnull().sum())


--- Missing Values Count ---
Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64


In [7]:
# 1. Drop rows where 'Rating' is missing
df_clean = df.dropna(subset=['Rating']).copy()

# 2. Fill missing values in text features with 'Unknown'
df_clean['Genre'] = df_clean['Genre'].fillna('Unknown')
df_clean['Director'] = df_clean['Director'].fillna('Unknown')
df_clean['Actor 1'] = df_clean['Actor 1'].fillna('Unknown')

# 3. Verify that there are no missing values left in these columns
print("--- Missing Values After Cleaning ---")
print(df_clean[['Rating', 'Genre', 'Director', 'Actor 1']].isnull().sum())
print(f"\nRemaining rows for training: {len(df_clean)}")

--- Missing Values After Cleaning ---
Rating      0
Genre       0
Director    0
Actor 1     0
dtype: int64

Remaining rows for training: 7919


In [8]:
# 1. Define the Feature matrix (Inputs)
X = df_clean[['Genre', 'Director', 'Actor 1']]

# 2. Define the Target vector (Output)
y = df_clean['Rating']

# 3. Print shapes to inspect dimensions
print("Feature matrix (X) shape:", X.shape)
print("Target vector (y) shape:", y.shape)

Feature matrix (X) shape: (7919, 3)
Target vector (y) shape: (7919,)


In [9]:
from sklearn.preprocessing import TargetEncoder

# 1. Initialize the Target Encoder
encoder = TargetEncoder(smooth="auto")

# 2. Fit the encoder on X and transform it into numerical values
# Note: TargetEncoder requires both X and y to calculate average ratings
X_encoded = encoder.fit_transform(X, y)

# 3. Inspect the encoded numeric array
print("Encoded X shape:", X_encoded.shape)
print("First 3 rows of encoded numerical data:")
print(X_encoded[:3])

Encoded X shape: (7919, 3)
First 3 rows of encoded numerical data:
[[6.3741439  5.85019732 6.7       ]
 [5.67118811 5.84604578 6.40556034]
 [6.236575   5.59834018 4.98242095]]


In [10]:
from sklearn.model_selection import train_test_split

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Verify the sizes of each set
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape:  {X_test.shape}")
print(f"Training target shape:   {y_train.shape}")
print(f"Testing target shape:    {y_test.shape}")

Training features shape: (6335, 3)
Testing features shape:  (1584, 3)
Training target shape:   (6335,)
Testing target shape:    (1584,)


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Initialize the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 2. Train (fit) the model on the training set
model.fit(X_train, y_train)

# 3. Generate predictions for the test set
y_pred = model.predict(X_test)

# 4. Evaluate performance metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE):  {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score:                  {r2:.2f}")

Mean Squared Error (MSE):  1.53
Root Mean Squared Error (RMSE): 1.24
R² Score:                  0.18


In [12]:
import joblib

joblib.dump(model, 'movie_rating_model.pkl')
joblib.dump(encoder, 'target_encoder.pkl')
print("Model and encoder saved successfully!")

Model and encoder saved successfully!


In [ ]:
!streamlit run app.py